# 4. Dashboard visuals (pipeline step 9)

This notebook runs **pipeline step 9** (`9_dashboard_visuals`). It **prebuilds all dashboard visualization artifacts** (BupaR, DTW, FP-Growth) on **EC2** and **saves them to S3** for **direct dashboard integration**. The dashboard loads these prebuilt assets from S3 (the API returns only URLs; no computation at request time). Visuals are **SHAP/FFA-driven**: model data and feature lists come from Step 3b / 7 / 8 so process mining and itemset mining use only important features.

**Flow:** Run after [3_model_train_shap_ffa.ipynb](3_model_train_shap_ffa.ipynb). Then run [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb) once to build and deploy.

## Steps

1. **Setup** – Resolve paths (scripts in `9_dashboard_visuals/`; outputs under `10_risk_dashboard/visualizations/{bupar,dtw,fpgrowth}/`).
2. **BupaR** – Process mining sequences and plots (SHAP/FFA allowed codes when available); **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/`.
3. **DTW** – Trajectory features and plots **based on SHAP/FFA important codes** (same as BupaR/FP-Growth); plot PNGs are **uploaded to the dashboard bucket** under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/`. The DTW tab includes **appointments vs no appointments** visuals: **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (outcome rate by trajectory intensity / archetype). These visuals use the **full pipeline (2016–2019)**: model_events (Step 4) and DTW features are built from all years 2016–2019, not a single year. **Extreme-density cohorts** (optional) support the same routine vs no routine comparison for high-utilizer subgroups—see optional step below.
4. **FP-Growth** – Itemsets, rules, **Plotly network HTML**, and PNGs; **uploaded to the dashboard bucket** (`S3_DASHBOARD_BUCKET`, e.g. jerome-dixon.io) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard loads the **network plot by cohort** from these URLs.
5. **Model performance metrics and cohort metadata** – Prebuilt via `generate_metrics.py` and `generate_metadata.py` (no recomputation). Deploy (5_build_and_deploy) uploads to the dashboard bucket: `metadata/model_performance_metrics.json` (Documentation tab) and `metadata/opioid_ed.json`, `metadata/non_opioid_ed.json` (dropdowns). Frontend loads these same-origin; Lambda GET /metrics and GET /metadata are fallbacks.
6. **API** – Returns URLs to prebuilt S3 assets only (no server-side computation for visuals).

Idempotent. Run from repo root. Prerequisites: notebook 5 done (`4_model_data`, `7_shap_analysis`, `8_ffa_analysis`); R and bupaR for BupaR.

In [1]:
# Setup: paths (outputs under 10_risk_dashboard/visualizations/)
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if (REPO_ROOT / "py_helpers").exists():
    pass  # already repo root
else:
    for p in REPO_ROOT.parents:
        if (p / "py_helpers").exists():
            REPO_ROOT = p
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Creation code (step 9); outputs go to 10_risk_dashboard/visualizations
STEP9_ROOT = REPO_ROOT / "9_dashboard_visuals"
VISUAL_ROOT = REPO_ROOT / "10_risk_dashboard" / "visualizations"
BUPAR_VISUALS_SCRIPT = STEP9_ROOT / "bupar" / "create_bupar_visuals.py"
DTW_FEATURES_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_features.py"
DTW_VISUALS_SCRIPT = STEP9_ROOT / "dtw" / "create_dtw_visuals.py"
FPGROWTH_VISUALS_SCRIPT = STEP9_ROOT / "fpgrowth" / "create_fpgrowth_visuals.py"

print(f"Repo root: {REPO_ROOT}")
print(f"Step 9 (scripts): {STEP9_ROOT}")
print(f"Outputs: {VISUAL_ROOT}")

Repo root: /home/pgx3874/pgx-analysis
Step 9 (scripts): /home/pgx3874/pgx-analysis/9_dashboard_visuals
Outputs: /home/pgx3874/pgx-analysis/10_risk_dashboard/visualizations


## Config: cohorts and age bands

In [2]:
from py_helpers.constants import COHORT_NAMES, AGE_BANDS

try:
    from py_helpers.constants import REQUIRED_COHORTS
except ImportError:
    _all_bands = ['0-12', '13-24', '25-44', '45-54', '55-64', '65-74', '75-84', '85-114']
    REQUIRED_COHORTS = {"opioid_ed": _all_bands, "non_opioid_ed": _all_bands}

COHORTS_TO_RUN = []
AGE_BANDS_TO_RUN = []

# Default: pipeline (cohort, age_band) from REQUIRED_COHORTS (both cohorts use full age bands 0-12 through 85-114)
if not COHORTS_TO_RUN and not AGE_BANDS_TO_RUN:
    combinations = [(c, ab) for c, bands in REQUIRED_COHORTS.items() for ab in bands]
    print("Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)")
else:
    if not COHORTS_TO_RUN:
        COHORTS_TO_RUN = COHORT_NAMES.copy()
    if not AGE_BANDS_TO_RUN:
        AGE_BANDS_TO_RUN = AGE_BANDS.copy()
    combinations = [(c, ab) for c in COHORTS_TO_RUN for ab in AGE_BANDS_TO_RUN]

print(f"Cohorts: {COHORTS_TO_RUN if COHORTS_TO_RUN else list(REQUIRED_COHORTS.keys())}")
print(f"Age bands: {AGE_BANDS_TO_RUN if AGE_BANDS_TO_RUN else 'per-cohort (REQUIRED_COHORTS)'}")
print(f"Total: {len(combinations)} combinations")

# Idempotent: skip cohort/age_band when output exists. Set FORCE_RERUN=True to re-run all.
FORCE_RERUN = False
# Parallel workers: BupaR and DTW can use many; FP-Growth limited for memory.
PARALLEL_WORKERS = 32
FPGROWTH_WORKERS = 4

Using pipeline-supported cohort/age_band (REQUIRED_COHORTS)
Cohorts: ['opioid_ed', 'non_opioid_ed']
Age bands: per-cohort (REQUIRED_COHORTS)
Total: 16 combinations


## Run BupaR process mining

Plots are uploaded to the dashboard bucket under `{S3_DASHBOARD_PREFIX}/bupar/{cohort}/{age_band}/plots/` (same pattern as FP-Growth). **BupaR features are not added to model data** (same as DTW and FP-Growth); they are for dashboard visualization only.

In [3]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

FAIL_FAST = True
FORCE_RERUN = True
force_flag = ["--force"] if FORCE_RERUN else []

def run_bupar_one(cohort_name, age_band):
    return (cohort_name, age_band, subprocess.run(
        [sys.executable, str(BUPAR_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=False,
    ).returncode)

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_bupar_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code = fut.result()
        print(f"  [BupaR] {cohort_name} / {age_band} -> exit {code}")
        if code != 0 and FAIL_FAST:
            raise RuntimeError(f"BupaR failed: {cohort_name} / {age_band}")
print("BupaR done.")

2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - Runtime environment: os=linux logical_cores=32 ram_gb=1024 fast_root=/mnt/nvme
2026-02-16 00:10:23,195 - INFO - [FUNCTION][5_bupar][create_bupar_visuals] START time=2026-02-16 00:10:23 mem_mb=2075.1 cpu_pct=0.0
2026-02-16 00:10:23,195 - INFO - [FUNCTION][5_bupar][create_bupar_visuals] START tim

## Run DTW trajectory features

**Trajectories are based on SHAP/FFA results** (same as BupaR/FP-Growth): `create_dtw_features.py` uses `get_shap_ffa_allowed_codes_combined()` to restrict trajectory events to SHAP/FFA important codes when available; if that is missing or empty, it falls back to all events in model_data. Run after Step 3b / 7 / 8 so SHAP/FFA outputs exist.

For each cohort/age band this step: (1) runs `create_dtw_features.py`, (2) runs `create_dtw_visuals.py` (publish: copy, S3, dashboard). DTW features are **not** added to model data; they are standalone for dashboard visuals. **DTW plot PNGs** are uploaded to the **dashboard bucket** (same as FP-Growth/BupaR) under `{S3_DASHBOARD_PREFIX}/dtw/{cohort}/{age_band}/plots/` by create_dtw_visuals.

- **Expected filenames** (dashboard S3): `dtw_trajectory_analysis_{cohort}_{age_band}.png`, `dtw_sample_trajectories_{cohort}_{age_band}.png` (use underscore in age band, e.g. `0_12`). **chart_data.json** (routine_comparison, high_risk_trajectories) is also prebuilt and uploaded for direct dashboard integration.
- **Local paths** from which plots are uploaded (first existing wins):  
  `10_risk_dashboard/visualizations/dtw/outputs/{cohort}/{age_band_fname}/plots/` or  
  `10_risk_dashboard/visualizations/dtw/outputs/{cohort}/{age_band}/plots/`.  
  Place PNGs in one of these so the add step uploads them; if no plots exist, upload is skipped.

The dashboard **DTW tab** shows **appointments vs no appointments**–related visuals: **Routine vs No Routine (Outcomes)** uses the **admin ICD filter** (`1b_apcd_event_filter/administrative_codes_lookup.json`): outcome rate for "No routine appointments (0 admin ICD events)" vs "Routine appointments (1+ admin ICD events)". **High-Risk vs Low-Risk Trajectories** shows outcome by trajectory archetype. DTW features include `admin_icd_event_count` (computed from model_events in create_dtw_features). **Date scope:** full pipeline (2016–2019).

In [ ]:
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True  # Stop on first failure; set False to continue (also in config/BupaR cell)
force_flag = ["--force"] if FORCE_RERUN else []

def run_dtw_one(cohort_name, age_band):
    # Capture output so we can print logs when a cohort fails (also see logs/feature_engineering/dtw/*.log)
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", cohort_name, "--age_band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    if r1.returncode != 0:
        return (cohort_name, age_band, r1.returncode, None, r1.stdout, r1.stderr, None, None)
    r2 = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=True,
        text=True,
    )
    return (cohort_name, age_band, r1.returncode, r2.returncode, r1.stdout, r1.stderr, r2.stdout, r2.stderr)

with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
    futures = {ex.submit(run_dtw_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, c1, c2, out1, err1, out2, err2 = fut.result()
        print(f"  [DTW] {cohort_name} / {age_band} -> exit {c1}, {c2}")
        if c1 != 0:
            ab_f = age_band.replace("-", "_")
            print(f"    create_dtw_features failed (exit {c1}). Check logs/feature_engineering/dtw/create_dtw_features_{cohort_name}_{ab_f}.log")
            if err1:
                print("    stderr:", (err1[:1500] + "..." if len(err1) > 1500 else err1))
            if out1:
                print("    stdout:", (out1[:800] + "..." if len(out1) > 800 else out1))
        if c2 is not None and c2 != 0:
            print("    create_dtw_visuals failed (exit %s)" % c2)
            if err2:
                print("    stderr:", (err2[:1500] + "..." if len(err2) > 1500 else err2))
        if c2 is None and FAIL_FAST:
            raise RuntimeError(f"DTW create_dtw_features failed: {cohort_name} / {age_band}")
        if c2 is not None and c2 != 0 and FAIL_FAST:
            raise RuntimeError(f"DTW create_dtw_visuals failed: {cohort_name} / {age_band}")
print("DTW done.")

### Appointments vs no appointments and extreme-density cohorts

**Research question (N1):** Is there a difference in outcomes for patients without routine appointments vs those with routine care? The DTW tab answers this via **Routine vs No Routine (Outcomes)** and **High-Risk vs Low-Risk Trajectories** (see above). These are shown over the **full pipeline (2016–2019)**, not a single year.

**Extreme-density cohorts** are high-utilizer patients (top ~5% by medical_code transaction density) split out so they do not dominate main models (see `docs/Step4_ModelData/README_model_data_and_extreme_split.md`). For **each cohort and age band**, running extract + DTW (and optionally BupaR) for the extreme-density subgroup lets you compare **routine vs no routine** (outcomes and trajectories) in the high-utilizer subgroup and how **extreme densities** and **extreme-density trajectories** differ across age bands and cohorts. By default the cell below uses the **same (cohort, age_band) combinations** as the main pipeline. Set `EXTREME_COMBINATIONS = []` to skip. Requires **Step 4** (model data) first.

In [ ]:
# Default: same (cohort, age_band) as main pipeline so we get routine vs no routine and
# extreme-density trajectories for every cohort and age band. Set to [] to skip.
EXTREME_COMBINATIONS = combinations  # from config cell above

EXTREME_EXTRACT_SCRIPT = STEP9_ROOT / "dtw" / "extract_extreme_density_cohort.py"
extreme_force_flag = ["--force"] if FORCE_RERUN else []

def run_extreme_one(cohort_name, age_band):
    r0 = subprocess.run(
        [sys.executable, str(EXTREME_EXTRACT_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band],
        cwd=str(REPO_ROOT),
        capture_output=False,
    )
    if r0.returncode != 0:
        return (cohort_name, age_band, r0.returncode, None, None)
    extreme_name = f"{cohort_name}_extreme_density"
    r1 = subprocess.run(
        [sys.executable, str(DTW_FEATURES_SCRIPT), "--cohort", extreme_name, "--age_band", age_band] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=False,
    )
    if r1.returncode != 0:
        return (cohort_name, age_band, r0.returncode, r1.returncode, None)
    r2 = subprocess.run(
        [sys.executable, str(DTW_VISUALS_SCRIPT), "--cohort-name", extreme_name, "--age-band", age_band,
         "--project-root", str(REPO_ROOT)] + extreme_force_flag,
        cwd=str(REPO_ROOT),
        capture_output=False,
    )
    return (cohort_name, age_band, r0.returncode, r1.returncode, r2.returncode)

if not EXTREME_COMBINATIONS:
    print("EXTREME_COMBINATIONS is empty; skipping extreme-density cohort extraction and DTW.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    with ThreadPoolExecutor(max_workers=PARALLEL_WORKERS) as ex:
        futures = {ex.submit(run_extreme_one, c, ab): (c, ab) for c, ab in EXTREME_COMBINATIONS}
        for fut in as_completed(futures):
            cohort_name, age_band, c0, c1, c2 = fut.result()
            print(f"  [Extreme] {cohort_name} / {age_band} -> extract={c0}, dtw_feat={c1}, dtw_vis={c2}")
            if c0 != 0 and FAIL_FAST:
                raise RuntimeError(f"Extract extreme cohort failed: {cohort_name} / {age_band}")
            if c1 is not None and c1 != 0 and FAIL_FAST:
                raise RuntimeError(f"DTW create_dtw_features failed: {cohort_name}_extreme_density / {age_band}")
            if c2 is not None and c2 != 0 and FAIL_FAST:
                raise RuntimeError(f"DTW create_dtw_visuals failed: {cohort_name}_extreme_density / {age_band}")
    print(f"Done: extreme-density extract + DTW for {len(EXTREME_COMBINATIONS)} combinations (parallel).")

## Run FP-Growth (itemsets, Plotly network HTML, S3 upload)

FP-Growth uses **SHAP/FFA-refined** model data: inputs come from `4_model_data` (built from Step 3b `cohort_feature_importance.csv`). For each cohort/age band this step: (1) ensures itemsets exist, (2) creates PNGs and **Plotly interactive network HTML**, (3) **uploads to the dashboard bucket** (e.g. `jerome-dixon.io`) under `{S3_DASHBOARD_PREFIX}/fpgrowth/{cohort}/{age_band}/plots/` (e.g. `vcu/pgx-risk-calculator/fpgrowth/...`). The dashboard then shows the **network plot for the user-selected cohort** via the `/visualizations/fpgrowth` API. **FP-Growth features are not added to model data** (same as DTW); they are for dashboard visualization only.

Run the cell below in parallel (FPGROWTH_WORKERS at a time; builds itemsets, Plotly HTML, uploads to S3).

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

try:
    FAIL_FAST
except NameError:
    FAIL_FAST = True
try:
    FPGROWTH_WORKERS
except NameError:
    FPGROWTH_WORKERS = 4
force_flag = ["--force"] if FORCE_RERUN else []

def run_fpgrowth_one(cohort_name, age_band):
    r = subprocess.run(
        [sys.executable, str(FPGROWTH_VISUALS_SCRIPT), "--cohort-name", cohort_name, "--age-band", age_band] + force_flag,
        cwd=str(REPO_ROOT),
        capture_output=False,
    )
    return (cohort_name, age_band, r.returncode)

with ThreadPoolExecutor(max_workers=FPGROWTH_WORKERS) as ex:
    futures = {ex.submit(run_fpgrowth_one, c, ab): (c, ab) for c, ab in combinations}
    for fut in as_completed(futures):
        cohort_name, age_band, code = fut.result()
        print(f"  [FP-Growth] {cohort_name} / {age_band} -> exit {code}")
        if code != 0 and FAIL_FAST:
            raise RuntimeError(f"FP-Growth failed: {cohort_name} / {age_band}")
print("FP-Growth done.")

## API (reference)

Lambda receives **user input** (cohort, age_band, model/feature selections) and **filters** only—it does not process or generate visualization data. All BupaR, DTW, and FP-Growth visuals are **prebuilt on EC2** and **saved to S3**; the API returns **URLs** to those prebuilt assets (filtered by cohort/age_band). Endpoints: `GET /visualizations/causal`, `/visualizations/bupar`, `/visualizations/dtw`, `/visualizations/fpgrowth`. See `10_risk_dashboard/backend/README.md`.

In [ ]:
print("Dashboard endpoints: 10_risk_dashboard/backend/README.md")
print("API Gateway deploy: utility_scripts/create_api_gateway_pgx_risk_calculator.sh")

## Next: Build and deploy

Build and deploy run **only** in [5_build_and_deploy.ipynb](5_build_and_deploy.ipynb). Run that notebook after this one.

In [ ]:
# Build and deploy run only in 5_build_and_deploy.ipynb. Run that notebook after this one.
print("Build and deploy (once): open 5_build_and_deploy.ipynb and run it after this notebook.")

*(Build and deploy — including frontend sync to S3 — are done only in notebook 3. See above.)*